
# 09 期限结构与换月：同一商品为什么有两本账

> **本章要解决的问题**：同一种螺纹钢，为什么 1 月合约和 5 月合约价格不同？研究图里连续的“螺纹钢价格”，为什么账户里赚不到？换月那天到底发生了什么？

## 本章导读

商品期货不是一个价格，而是一张**期限结构**：不同交割月的合约各有报价。研究趋势时，我们想看到一条连续的价格序列；做交易时，账户只认识具体合约。这两件事必须分开算。

本章分六节：

1. 期货曲线、持有成本、便利收益；
2. 基差、展期收益、carry；
3. 主力连续合约：换月、复权、两本账；
4. 固定期限插值；
5. 换月成本与展期；
6. 期限结构因子：斜率、曲率、carry。

### 学习目标

- 能从合约价格画期限结构，判断 contango/backwardation。
- 能算 roll yield、年化 carry、换月成本。
- 能处理合约乘数、最小变动、最后交易日。
- 能比较不同换月规则对回测的影响。
- 能解释便利收益/季节性对曲线的影响。

### 先修

第01章的乘数/成本；第08章的对数收益；会 pandas 的基础操作。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="09 期限结构与换月：同一商品的两本账" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>09 期限结构与换月：同一商品的两本账｜知识图</title>
<desc>本章主线：研究看连续曲线，交易只认具体合约；换月规则不同，回测的账就不同。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a05" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao05" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag05" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">09</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">期限结构与换月：同一商品的两本账</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：研究看连续曲线，交易只认具体合约；换月规则不同，回测的账就不同。</text>
<rect x="41" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="121" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">各月合约报价与规则</text>
<text x="121" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">乘数、最小变动、最后交易日</text>
<rect x="51" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="68" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输入</text>
<rect x="227" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="307" y="132.1" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">期限结构 · carry</text>
<text x="307" y="150.6" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">· roll yield</text>
<text x="307" y="170.6" font-size="10.5" fill="#6E7B8B" text-anchor="middle">基差、换月、固定期限插值</text>
<rect x="237" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="254" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">处理</text>
<rect x="413" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="493" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">换月表与连续序列</text>
<text x="493" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">研究图与账户各一本账</text>
<rect x="423" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="440" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输出</text>
<rect x="599" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="679" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">与实盘对账</text>
<text x="679" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">换月规则一致才可比</text>
<rect x="609" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="626" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">检查</text>
<path d="M 201 149 L 227 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a05)"/>
<path d="M 387 149 L 413 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a05)"/>
<path d="M 573 149 L 599 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a05)"/>
<rect x="41" y="232" width="718" height="152" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<rect x="58" y="246" width="4" height="11" fill="#E07B39" rx="1"/>
<text x="70" y="256" font-size="11.5" fill="#1B3A5C" font-weight="bold">同一品种的期限结构与两本账</text>
<path d="M 95 360 L 500 360" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag05)"/>
<path d="M 95 360 L 95 300" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag05)"/>
<text x="86" y="306" font-size="8.6" fill="#6E7B8B" text-anchor="end">价格</text>
<text x="504" y="376" font-size="8.6" fill="#6E7B8B">月份</text>
<line x1="130" y1="360" x2="130" y2="365" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="130" y="377" font-size="8.2" fill="#6E7B8B" text-anchor="middle">01</text>
<line x1="215" y1="360" x2="215" y2="365" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="215" y="377" font-size="8.2" fill="#6E7B8B" text-anchor="middle">05</text>
<line x1="300" y1="360" x2="300" y2="365" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="300" y="377" font-size="8.2" fill="#6E7B8B" text-anchor="middle">07</text>
<line x1="385" y1="360" x2="385" y2="365" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="385" y="377" font-size="8.2" fill="#6E7B8B" text-anchor="middle">10</text>
<line x1="470" y1="360" x2="470" y2="365" stroke="#6E7B8B" stroke-width="1" stroke-linecap="round"/>
<text x="470" y="377" font-size="8.2" fill="#6E7B8B" text-anchor="middle">01</text>
<path d="M 130 318 L 240 330 L 340 340 L 440 350 L 480 353" fill="none" stroke="#1B3A5C" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M 130 342 L 240 330 L 340 318 L 440 308 L 480 304" fill="none" stroke="#E07B39" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="6 4"/>
<line x1="540" y1="290" x2="560" y2="290" stroke="#1B3A5C" stroke-width="1.8" stroke-linecap="round"/>
<text x="566" y="293" font-size="8.4" fill="#1B3A5C">backwardation：近强远弱</text>
<line x1="540" y1="306" x2="560" y2="306" stroke="#E07B39" stroke-width="1.8" stroke-linecap="round" stroke-dasharray="5 3"/>
<text x="566" y="309" font-size="8.4" fill="#E07B39">contango：近弱远强</text>
<rect x="540" y="323" width="3" height="7" fill="#E07B39" rx="1"/>
<text x="548" y="330" font-size="8.4" fill="#6E7B8B">研究图：主力连续</text>
<rect x="540" y="338" width="3" height="7" fill="#E07B39" rx="1"/>
<text x="548" y="345" font-size="8.4" fill="#6E7B8B">账户：具体合约</text>
<rect x="540" y="353" width="3" height="7" fill="#E07B39" rx="1"/>
<text x="548" y="360" font-size="8.4" fill="#6E7B8B">换月规则不同 → 回测不同</text>
<rect x="540" y="368" width="3" height="7" fill="#E07B39" rx="1"/>
<text x="548" y="375" font-size="8.4" fill="#6E7B8B">两本账要能对上</text>
</svg>

> **图**：本章主线：研究看连续曲线，交易只认具体合约；换月规则不同，回测的账就不同。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：商品研究员、交易员、展期执行。
- **输入**：各月合约价格、合约规则、库存/仓单。
- **输出**：期限结构、roll yield、年化 carry、换月表。
- **在流程的哪一步**：期限结构因子、展期与基差交易。
- **失败模式**：合约乘数/最小变动、换月规则不一致、忽略流动性与交割。
- **最小工作流**：取曲线 → 算 carry/roll → 定换月规则 → 回测 → 与实盘对账。

---
## 1. 期货曲线、持有成本、便利收益

### 1.1 一张曲线

同一品种、同一时刻，不同交割月的价格连起来，叫期限结构：

| 合约 | 交割月 | 价格 |
|---|---|---:|
| rb2701 | 2027-01 | 3100 |
| rb2705 | 2027-05 | 3070 |
| rb2710 | 2027-10 | 3040 |

近月高、远月低，叫 backwardation（现货溢价/近强远弱）；近月低、远月高，叫 contango（近弱远强）。

### 1.2 持有成本

把现货存到未来，要付资金、仓储、保险；持有现货也可能带来便利（不断料、可交割）。简化模型：

$$F_T=S_0e^{(r+u-y)T},$$

**公式拆解**
- **符号**：$S_0$ 现货（元/吨），$F_T$ 到期期货价，$r$ 资金、$u$ 仓储、$y$ 便利（1/年），$T$ 年。
- **结构**：$F_T=S_0e^{(r+u-y)T}$，净持有成本 $c=r+u-y$；$c>0$ 为 contango。
- **代入**：$S_0=100,c=0.04,T=0.5$：$F=102.02$，年化斜率 4%。
- **边界**：$T\to0$ 时 $F\to S_0$（基差收敛）。
- **用途**：判曲线形状、算 carry 的锚；$F_T$ 不是未来现货预测。

- $r$：资金成本；
- $u$：仓储/保险；
- $y$：便利收益。

净持有成本 $c=r+u-y$。$c>0$ 时远月贵（contango）；$c<0$ 时近月贵（backwardation）。

### 1.3 手算

现货 100，净持有成本年率 4%，半年：

$$F=100e^{0.04\times0.5}\approx102.02.$$

反推年化斜率：

$$\frac{\ln(102.02/100)}{0.5}\approx0.04.$$

### 1.4 代码：年化 carry



In [2]:

import numpy as np

near, far, days = 3100.0, 3070.0, 90
carry = 365 / days * np.log(near / far)
print("年化 carry", round(carry, 4))


年化 carry 0.0394



**读数与应用**：近月 3100、远月 3070，carry 约 +3.9%：近月比远月贵，做空近月/做多远月可以收 carry（忽略成本）。符号取决于“近减远”还是“远减近”，报告要写清。

**常见错误**：

1. 把 contango/backwardation 的方向说反。
2. 用简单收益率代替对数收益率做年化。
3. 忽略便利收益；现实曲线不是纯持有成本。

**停下来想一想**：为什么现实中的期限结构不会严格贴住持有成本曲线？

<details><summary>对照你的思路</summary>

仓储有限、品质差异、交割规则、运输、季节性、便利收益变化、资金成本不同，都会让曲线偏离。理论曲线是组织思路的基准，不是精确预测。

</details>

---



## 2. 基差、展期收益、carry

### 2.1 基差

$$\text{basis}=S-F \quad\text{或}\quad F-S.$$

现货减期货，还是期货减现货，要写清。基差变化反映“近远价差”的变化。

### 2.2 展期收益

持有近月、到期前滚到远月，收益有一部分来自价差：

$$\text{roll yield}\approx\frac{F_{\text{near}}-F_{\text{far}}}{F_{\text{near}}}\times\frac{365}{\Delta T}.$$

**公式拆解**
- **符号**：$F_{\text{near}},F_{\text{far}}$ 近远月价（元/吨），$\Delta T$ 到期差（天），$365/\Delta T$ 年化因子。
- **结构**：分子近减远（正为 backwardation），分母用近月价归一化。
- **代入**：3100/3070、90 天：$0.9677\%\times365/90=3.92\%$。
- **边界**：$\Delta T\to0$ 年化爆掉；符号约定要写清。
- **用途**：估展期成本、判多空；不是无风险收益。

若近月贵（backwardation），空近月/多远月可以收 roll；若 contango，持有多头要付 roll。

### 2.3 carry 因子

横截面上，比较不同品种的 carry：

$$\text{carry}_i=\frac{365}{\Delta T_i}\ln\frac{F_{i,\text{near}}}{F_{i,\text{far}}}.$$

**公式拆解**
- **符号**：$i$ 品种，$F_{\text{near}},F_{\text{far}}$ 近远月价，$\Delta T_i$ 期限差，$\text{carry}_i$ 年化对数 carry。
- **结构**：分子是近远月对数价差，乘 $365/\Delta T_i$ 年化。
- **代入**：近 3100、远 3070、90 天：$\ln(3100/3070)\times4.056=3.94\%$。
- **边界**：价格非正或 $\Delta T\to0$ 时失效。
- **用途**：横截面多高、空低；高 carry 不等于必涨。

高 carry 做多、低 carry 做空，是商品 carry 策略的常见构造。但要注意：不同品种的期限结构、季节性、交割规则不同。

### 2.4 代码：横截面 carry



In [3]:

import numpy as np
import pandas as pd

df = pd.DataFrame({
    "asset": ["A", "B", "C"],
    "near": [3100.0, 8100.0, 5100.0],
    "far": [3070.0, 8050.0, 5150.0],
    "days": [90, 180, 60],
})
df["carry"] = 365 / df["days"] * np.log(df["near"] / df["far"])
print(df.sort_values("carry", ascending=False).round(4))


  asset    near     far  days   carry
0     A  3100.0  3070.0    90  0.0394
1     B  8100.0  8050.0   180  0.0126
2     C  5100.0  5150.0    60 -0.0594



**读数与应用**：A、B 的 near 高于 far，carry 为正；C 的 near 低于 far，carry 为负。排序可以做多空，但样本只有 3 个品种，不能直接交易。

**常见错误**：

1. 不同品种用不同“近/远”定义，carry 不可比。
2. 不年化，直接比原始价差。
3. 忽略换月/交割规则；有些品种的近月流动性很差。

**停下来想一想**：carry 高，是否一定该做多？

<details><summary>对照你的思路</summary>

不一定。carry 高可能来自近月挤仓/流动性溢价，也可能是供应紧张的基本面；价格可能继续涨，也可能突然反转。carry 是一个横截面信号，要配合风险、成本、季节性、库存等因素。

</details>

---



## 3. 主力连续合约：换月、复权、两本账

### 3.1 为什么会有“假跳空”

研究图常用“主力连续”价格：今天用 rb2701，明天换到 rb2705。两个合约价格不同，拼接处会出现跳空：

```
旧合约：98 → 100
新合约：       103 → 104
拼接后：98 → 100 → 103 → 104
```

这个 +3 的跳空不是行情，是换月。

### 3.2 两种复权

- **差值复权**：把旧序列平移，使接缝对齐；
- **比率复权**：按比例缩放，使接缝对齐。

差值复权保留“点差”，比率复权保留“百分比”。用哪个，取决于研究的是绝对点数还是收益率。

### 3.3 两本账

- **研究账**：连续价格/复权序列，用来算因子、画图；
- **交易账**：实际合约的成交价、乘数、手续费、换月成本。

**研究账的盈亏不等于交易账的盈亏。** 换月时，旧合约平仓、新合约开仓，两段盈亏要分别算。

### 3.4 代码：换月两段盈亏



In [4]:

old = [98.0, 100.0]
new = [103.0, 104.0]
mult = 10

gross = ((old[1] - old[0]) + (new[1] - new[0])) * mult
print("真实两段毛利", gross)
print("用未复权连续价误算", (new[1] - old[0]) * mult)


真实两段毛利 30.0
用未复权连续价误算 60.0



**读数与应用**：真实两段毛利 30 元；用未复权连续价 98→104 会算成 60 元，多算 30 元。乘数 10，误差被放大 10 倍。

**常见错误**：

1. 用未复权连续价算策略收益。
2. 用复权价当成交价。
3. 换月日用了“当天收盘后才知道”的持仓量排名。

**停下来想一想**：差值复权和比率复权，哪个更适合做收益率因子？

<details><summary>对照你的思路</summary>

比率复权保留百分比变化，更适合收益率/动量因子；差值复权保留绝对点差，更适合做点位/价差研究。两者都会改变历史，必须保留原始合约价格和复权因子。

</details>

---



## 4. 固定期限插值

### 4.1 为什么需要

今天比较“剩 15 天”和“剩 75 天”的合约，明天可能变成“剩 14 天”和“剩 74 天”；观察对象在变。研究期限结构时，要固定“剩余期限”，比如永远看 30 天、60 天、90 天。

### 4.2 线性插值

给定两个实际合约：

- 近月剩 $d_1$ 天，价格 $P_1$；
- 远月剩 $d_2$ 天，价格 $P_2$。

要插 $d\in[d_1,d_2]$：

$$P(d)=wP_1+(1-w)P_2,\qquad w=\frac{d_2-d}{d_2-d_1}.$$

**公式拆解**
- **符号**：$d$ 目标期限（天），$d_1,d_2$ 近远月期限，$P_1,P_2$ 价格，$w$ 近月权重。
- **结构**：$P(d)=wP_1+(1-w)P_2$，$w=(d_2-d)/(d_2-d_1)$。
- **代入**：$d_1=20,d_2=80,P_1=100,P_2=106$：$d=35$ 时 $w=0.75$，$P=101.5$。
- **边界**：$d$ 超出 $[d_1,d_2]$ 是外推，不能当内插。
- **用途**：构造 30/60/90 天固定期限序列；不是可成交价。

$d$ 越接近近月，$w$ 越大。

### 4.3 外推

$d$ 超出 $[d_1,d_2]$ 时，不能硬插；要显式外推，或者换一对合约。外推的误差更大，报告要标注。

### 4.4 插值价不是可成交价

插值价是研究用的“合成价”，没有订单簿。真正持仓还要映射到具体合约和权重。

### 4.5 代码：固定期限插值



In [5]:

def constant_maturity(days, near_days=20, far_days=80, near_price=100.0, far_price=106.0):
    if not near_days <= days <= far_days:
        return None
    weight = (far_days - days) / (far_days - near_days)
    return weight * near_price + (1 - weight) * far_price

for days in [35, 50, 100]:
    print("剩余", days, "天，插值价", constant_maturity(days))


剩余 35 天，插值价 101.5
剩余 50 天，插值价 103.0
剩余 100 天，插值价 None



**读数与应用**：35 天权重 0.75，价格 101.5；50 天权重 0.5，价格 103；100 天超出区间，返回 None。实盘要记录用的哪一对合约、插值方法、外推规则。

**常见错误**：

1. 用价格插值，却用对数价格讨论百分比；口径要统一。
2. 外推不标注，把合成价当真实价。
3. 插值权重每天变，却把合成价当“固定持仓”。

**停下来想一想**：固定期限的合成持仓，每天需要调仓吗？

<details><summary>对照你的思路</summary>

需要。剩余期限每天变，插值权重也变；即使不换合约，合成价的变化也包含“权重变化”。要声称组合盈亏，必须按前一期已知权重算分腿收益，再计再平衡成本。

</details>

---



## 5. 换月成本与展期

### 5.1 换月成本

换月一次，至少付：

- 旧合约平仓：价差/滑点/手续费；
- 新合约开仓：价差/滑点/手续费；
- 两腿的冲击成本。

### 5.2 换月日

主力合约切换不能“当天收盘后才知道当天该换”。要用**前一日收盘后**已知的持仓量/成交量排名，决定下一交易日换月；否则是未来函数。

### 5.3 展期安排

- 固定日期换月（比如交割前 10 天）；
- 按持仓量排名换月；
- 按流动性/价差换月。

不同规则会改变回测结果，报告要写清。

### 5.4 代码：换月一次的成本



In [6]:

spread_points, mult, fee, slippage_points = 1.0, 10, 6, 0.5

cost = spread_points * mult + 2 * fee + slippage_points * mult
print("换月一次成本", cost, "元")


换月一次成本 27.0 元



**读数与应用**：价差 10 元 + 手续费 12 元 + 滑点 5 元 = 27 元/手。若每月换一次，一年 12 次，成本 324 元/手；对 1 万元账户、1 手螺纹钢，这不是小数目。

**常见错误**：

1. 只算手续费，不算价差/滑点。
2. 换月日用了当天收盘后的排名。
3. 不考虑新旧合约流动性，按中间价成交。

**停下来想一想**：为什么“主力连续”回测容易高估收益？

<details><summary>对照你的思路</summary>

主力连续把不同合约拼成一条平滑序列，忽略了换月的价差、手续费、滑点；还可能用“当天收盘后才知道的主力”决定当天换月，偷看未来。研究图可以复权，交易账必须逐合约算。

</details>

---



## 6. 期限结构因子：斜率、曲率、carry

### 6.1 三个形状

- **斜率**：近月与远月的差，$\frac{F_{\text{far}}-F_{\text{near}}}{F_{\text{near}}}$；
- **曲率**：中间合约相对两端的弯曲，$\frac{F_{\text{near}}-2F_{\text{mid}}+F_{\text{far}}}{F_{\text{near}}}$；
- **carry**：年化近远价差。

### 6.2 经济含义

- 斜率为正（contango）：远月贵，持有多头要付 roll；
- 斜率为负（backwardation）：近月贵，多头可能收 roll；
- 曲率：反映中间月份的供需/季节性。

### 6.3 横截面策略

每个品种算一个 carry/斜率，排序做多高、做空低。要注意：

- 品种间期限结构不可直接比，要标准化；
- 季节性/库存/交割规则；
- 换月成本；
- 极端行情下，carry 会反转。

### 6.4 代码：斜率与曲率



In [7]:

near, mid, far = 3100.0, 3050.0, 3040.0

slope = (far - near) / near
curvature = (near - 2 * mid + far) / near
print("斜率", round(slope, 4))
print("曲率", round(curvature, 4))


斜率 -0.0194
曲率 0.0129



**读数与应用**：斜率为负（backwardation），曲率为正（中间略低/两端高）。这两个数与 carry 一起，构成期限结构的三个基本特征。

**常见错误**：

1. 不同品种用不同合约对，斜率不可比。
2. 不标准化，直接比较绝对价格差。
3. 忽略换月/流动性，因子无法交易。

**停下来想一想**：carry 因子和动量因子有什么区别？

<details><summary>对照你的思路</summary>

动量看“过去价格涨跌”，carry 看“期限结构给出的持有收益”。两者可能同向，也可能相反：一个品种过去涨得多（动量高），但近月远月价差显示 contango（carry 负），这时两个因子会打架。多因子组合要处理相关性。

</details>

---



## 本章小结

### 一张表

| 概念 | 公式/做法 | 用途 |
|---|---|---|
| 持有成本 | $F=Se^{(r+u-y)T}$ | 理论曲线 |
| 基差 | $S-F$ 或 $F-S$ | 近远价差 |
| carry | $\frac{365}{\Delta T}\ln(F_{\text{near}}/F_{\text{far}})$ | 横截面因子 |
| 差值复权 | 平移旧序列 | 保留点差 |
| 比率复权 | 按比例缩放 | 保留百分比 |
| 固定期限 | 线性插值 | 可比期限 |
| 换月成本 | 价差+手续费+滑点 | 净收益 |
| 斜率 | $(F_{\text{far}}-F_{\text{near}})/F_{\text{near}}$ | 期限结构 |
| 曲率 | $(F_{\text{near}}-2F_{\text{mid}}+F_{\text{far}})/F_{\text{near}}$ | 弯曲 |

### 八句话

1. 期货不是一个价格，是一张曲线。
2. contango 远月贵，backwardation 近月贵。
3. carry 的符号取决于“近减远”还是“远减近”。
4. 连续合约的跳空是换月，不是行情。
5. 研究账可以复权，交易账必须逐合约。
6. 换月日不能用当天收盘后的排名。
7. 固定期限插值价没有订单簿。
8. 斜率/曲率/carry 是三个不同的期限结构特征。

---

## 本章习题

**1.** 近月 3100、远月 3070。这是 contango 还是 backwardation？

**2.** 近月 3100、远月 3070、90 天。求年化 carry。

**3.** 旧合约 98→100，新合约 103→104，乘数 10。求真实两段毛利；未复权连续价会误算成多少？

**4.** 差值复权和比率复权，哪个保留百分比变化？

**5.** 近月剩 20 天价格 100，远月剩 80 天价格 106。求剩 35 天的插值价。

**6.** 价差 1 点、乘数 10、每边手续费 6 元、滑点 0.5 点。求换月一次成本。

**7.** 近月 3100、中间 3050、远月 3040。求斜率和曲率。

**8.** 为什么“主力连续”回测容易高估收益？

<details><summary>参考答案</summary>

**1.** 近月高、远月低，backwardation。

**2.** $\frac{365}{90}\ln(3100/3070)\approx0.0395$，约 3.95%。

**3.** 真实 $(100-98+104-103)\times10=30$；未复权 $(104-98)\times10=60$，多算 30。

**4.** 比率复权保留百分比变化。

**5.** 权重 $(80-35)/60=0.75$，价格 $0.75\times100+0.25\times106=101.5$。

**6.** $1\times10+2\times6+0.5\times10=27$ 元。

**7.** 斜率 $(3040-3100)/3100\approx-0.0194$；曲率 $(3100-2\times3050+3040)/3100=40/3100\approx0.0129$。

**8.** 忽略换月价差/手续费/滑点，可能用当天收盘后的主力排名偷看未来。

</details>

---

## 参考资料

- Hull, J. C. *Options, Futures, and Other Derivatives*（期限结构、持有成本）。
- Gorton, G. & Rouwenhorst, K. G. *Facts and Fantasies about Commodity Futures*（商品 carry/roll）。
- 华泰期货《商品中的水平、斜率、曲率因子》（期限结构因子）。
